# 01 — Petrol prices vs CNG price scenarios

**Question:** how big is the gap between what drivers pay for petrol and what they pay for CNG?

This notebook:
1. Loads the NBS national average petrol price (₦ per litre, monthly) from `data/raw/petrol_prices_nbs.csv`
2. Overlays the three CNG price scenarios (₦230 / ₦318 / ₦380 per scm) from `data/raw/model_inputs.csv`
3. Saves an interactive chart to `outputs/figures/petrol_vs_cng.html`

**Sources** (see `docs/sources.md`): petrol: NBS PMS Price Watch, as reported in S14, S15, S16, S30, S31, S32.
CNG: S13 (₦230, NNPC stations Sep 2025), S11 (₦318, BusinessDay Oct 2026), S10 (₦380, official NGML/NADDC rate Aug 2026). See conflict C7 in `docs/data_conflicts.md`.

**Units caveat:** petrol is priced per **litre**, CNG per **standard cubic metre (scm)**. These are different units, so the
chart puts them on a common naira axis but is *not* an energy-equivalent comparison. The project treats
1 scm ≈ 1 litre in distance terms (conflict C4: the CNG:petrol km ratio is about 1:1 across S20 and S05), which is why
the gap between the lines is a fair first-pass view of the per-km fuel saving.

## 1. Setup

We reuse `read_csv_safe` and `RAW_DIR` from `src/load_data.py` rather than calling `pd.read_csv` directly.
That way every notebook loads the raw files the same way (including the fix for unquoted commas in the
`notes` column), and there's one place to change if the loading logic changes.

`src/` isn't an installed package, so we add it to `sys.path` before importing.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

# Find the project root. Jupyter usually starts in notebooks/, but some editors start
# in the project root, so handle both instead of assuming one.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
import load_data  # noqa: E402  (import after the sys.path tweak, on purpose)

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the petrol series

`read_csv_safe` returns every column as text (it uses Python's `csv` module, which doesn't infer types),
so we convert explicitly: `month` → a datetime, prices → numbers. `errors="coerce"` turns blanks into `NaN`
instead of raising an error. The Lagos column is mostly blank, which is expected.

In [2]:
petrol = load_data.read_csv_safe(load_data.RAW_DIR / "petrol_prices_nbs.csv")

petrol["month"] = pd.to_datetime(petrol["month"], format="%Y-%m")
petrol["national_avg_ngn_per_litre"] = pd.to_numeric(petrol["national_avg_ngn_per_litre"], errors="coerce")

petrol = petrol.sort_values("month").reset_index(drop=True)
petrol[["month", "national_avg_ngn_per_litre", "source_id"]]

,month,national_avg_ngn_per_litre,source_id
0,2024-06-01,750.17,S30
1,2024-11-01,1214.17,S31
2,2024-12-01,1189.12,S32
3,2025-02-01,1245.80,S14
4,2025-04-01,1239.33,S15
5,2025-05-01,1027.76,S16
6,2025-06-01,1037.66,S30
7,2025-10-01,1052.31,S31
8,2025-11-01,1061.35,S31
9,2025-12-01,1048.63,S32


### Gaps in the series

We only have the months that press reports of NBS releases mention, not every month (the original NBS
PDFs are on the to-find list in `docs/sources.md`). If we drew a line straight through, say, Jul–Sep 2025,
the chart would *imply* data we don't have.

So we **reindex to a complete monthly calendar**: missing months become `NaN`, and plotly breaks the line
at each `NaN`. Markers show every real observation, including isolated ones like Jun 2024.

In [3]:
all_months = pd.date_range(petrol["month"].min(), petrol["month"].max(), freq="MS")  # MS = month start
petrol_monthly = petrol.set_index("month").reindex(all_months)
petrol_monthly.index.name = "month"

n_obs = petrol_monthly["national_avg_ngn_per_litre"].notna().sum()
print(f"{n_obs} observed months out of {len(all_months)} in the range "
      f"{all_months[0]:%b %Y} – {all_months[-1]:%b %Y}")

15 observed months out of 24 in the range Jun 2024 – May 2026


## 3. Load the CNG price scenarios

The three prices come from the `cng_price` row for cars/buses in `model_inputs.csv` (low / mid / high columns).
Reading them from the file instead of typing `230, 318, 380` keeps one source of truth: if the scenarios are
updated, this chart updates too. The `assert` is a guard that fails loudly if the file and the brief disagree.

In [4]:
inputs = load_data.read_csv_safe(load_data.RAW_DIR / "model_inputs.csv")
cng_row = inputs[(inputs["parameter"] == "cng_price") & (inputs["vehicle"] == "car/bus")].iloc[0]

cng_scenarios = {
    # label: (price in NGN per scm, source id from docs/sources.md)
    "Low":     (float(cng_row["low"]),  "S13"),
    "Central": (float(cng_row["mid"]),  "S11"),
    "High":    (float(cng_row["high"]), "S10"),
}
assert [p for p, _ in cng_scenarios.values()] == [230, 318, 380], cng_scenarios
cng_scenarios

{'Low': (230.0, 'S13'), 'Central': (318.0, 'S11'), 'High': (380.0, 'S10')}

## 4. Build the chart

Design choices, briefly:
- **One y-axis in naira.** Both fuels are priced in ₦, so they share an axis. The unit difference (litre vs scm) is spelled out in
  the axis title, the line labels and the hover text.
- **Petrol = solid blue line with markers**; **CNG = dashed orange reference lines**, all one colour because they are
  three scenarios of the *same* thing, told apart by their direct labels.
- **y-axis starts at 0.** The point of the chart is the *size* of the gap, and a truncated axis would exaggerate or hide it.
- **CNG lines are ordinary two-point traces**, not `fig.add_hline`. `add_hline` is a "shape", and shapes get no hover
  tooltip or legend entry. A trace from the left edge to the right edge of the x-axis looks the same and gets both.

In [5]:
PETROL_COLOR = "#2a78d6"   # blue
CNG_COLOR = "#eb6834"      # orange
INK_MUTED = "#52514e"      # grey for secondary text

fig = go.Figure()

# Petrol: national average, NGN per litre
fig.add_trace(go.Scatter(
    x=petrol_monthly.index,
    y=petrol_monthly["national_avg_ngn_per_litre"],
    name="Petrol: NBS national average (₦ per litre)",
    mode="lines+markers",
    line=dict(color=PETROL_COLOR, width=2),
    marker=dict(size=8, color=PETROL_COLOR, line=dict(color="white", width=2)),
    connectgaps=False,          # break the line where months are missing
    customdata=petrol_monthly["source_id"],
    hovertemplate="%{x|%b %Y}<br><b>₦%{y:,.2f} per litre</b><br>Source: %{customdata}<extra>Petrol</extra>",
))

# Fix the x-axis range (half a month of padding each side) so the CNG lines can
# span exactly edge to edge.
x0 = all_months[0] - pd.Timedelta(days=15)
x1 = all_months[-1] + pd.Timedelta(days=15)

# CNG: one dashed horizontal line per scenario, NGN per scm
for i, (label, (price, src)) in enumerate(cng_scenarios.items()):
    # Direct label in the right margin. The three lines sit close together, so
    # labels drawn on the plot overlap; nudging high up and low down keeps them apart.
    fig.add_annotation(
        text=f"CNG {label.lower()}: ₦{price:,.0f}/scm ({src})",
        xref="paper", x=1.01, xanchor="left",
        y=price, yref="y", yshift={"Low": -7, "Central": 0, "High": 7}[label],
        showarrow=False, font=dict(color=INK_MUTED, size=12),
    )
    # The line itself. showlegend only on the first, so the legend shows one entry for the group
    fig.add_trace(go.Scatter(
        x=[x0, x1], y=[price, price],
        mode="lines",
        line=dict(color=CNG_COLOR, width=2, dash="dash"),
        name="CNG price scenarios (₦ per scm)",
        legendgroup="cng",
        showlegend=(i == 0),
        hovertemplate=f"<b>CNG {label.lower()}: ₦{price:,.0f} per scm</b><br>Source: {src}<extra>CNG</extra>",
    ))

fig.update_layout(
    title=dict(
        text="Petrol vs CNG prices in Nigeria<br>"
             "<sup>Petrol: NBS national average, ₦ per litre, monthly (gaps = month not yet sourced). "
             "CNG: price scenarios, ₦ per standard cubic metre (scm).</sup>",
        x=0, xanchor="left",
    ),
    template="plotly_white",
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color="#0b0b0b"),
    xaxis=dict(title=dict(text="Month", standoff=8), range=[x0, x1], tickformat="%b %Y", dtick="M3", showgrid=False),
    yaxis=dict(
        title=dict(text="Price (₦): petrol per litre, CNG per scm", standoff=12),
        tickprefix="₦", tickformat=",.0f",
        rangemode="tozero", gridcolor="#ececea",
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0, xanchor="left"),
    hovermode="closest",
    margin=dict(t=130, l=90, r=210, b=130),
    height=600,
)

fig.add_annotation(
    text="Sources: NBS PMS Price Watch via S14, S15, S16, S30, S31, S32; CNG prices S10, S11, S13 (docs/sources.md).<br>"
         "Litre and scm are different units; ~1 scm ≈ 1 litre in distance terms (conflict C4).",
    xref="paper", yref="paper", x=0, y=-0.24, showarrow=False,
    xanchor="left", align="left", font=dict(size=11, color=INK_MUTED),
)

fig.show()

## 5. Save

`write_html` writes a standalone interactive page. `include_plotlyjs="cdn"` links to the plotly.js library online instead of
embedding a ~4 MB copy in every file, so the file stays small enough to commit. The trade-off: it needs an internet
connection to render. Switch to `include_plotlyjs=True` if you need it to work offline.

In [6]:
out_path = FIG_DIR / "petrol_vs_cng.html"
fig.write_html(out_path, include_plotlyjs="cdn")
print(f"Saved {out_path.relative_to(PROJECT_ROOT)}")

Saved outputs\figures\petrol_vs_cng.html


## 6. What the chart shows

The gap, computed rather than eyeballed. Using the saving-share approximation from `docs/model_inputs.md`
(saving ≈ 1 − CNG price ÷ petrol price, valid because 1 scm ≈ 1 litre in distance terms):

In [7]:
latest = petrol.dropna(subset=["national_avg_ngn_per_litre"]).iloc[-1]
latest_price = latest["national_avg_ngn_per_litre"]

rows = []
for label, (price, src) in cng_scenarios.items():
    rows.append({
        "CNG scenario": label,
        "CNG ₦/scm": price,
        f"Petrol ₦/L ({latest['month']:%b %Y}, {latest['source_id']})": latest_price,
        "Gap ₦": round(latest_price - price, 2),
        "Fuel saving share": f"{1 - price / latest_price:.0%}",
    })
pd.DataFrame(rows)

,CNG scenario,CNG ₦/scm,"Petrol ₦/L (May 2026, S16)",Gap ₦,Fuel saving share
0,Low,230.0,1596.25,1366.25,86%
1,Central,318.0,1596.25,1278.25,80%
2,High,380.0,1596.25,1216.25,76%
